        Disable GPU: Force tensorflow to select CPU

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="-1"    
import tensorflow as tf

2025-06-17 21:25:36.495298: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750175736.516076 3234924 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750175736.523358 3234924 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-17 21:25:36.545723: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


        Select a GPU with a memory limit

In [ ]:
#%% # include ../dirx 
import tensorflow as tf
print(f"TensorFlow version {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
len(gpus)
mylibpath = [
    '/home/kishoretarafdar/bin',
    # '/data1/kishoretarafdar/src2D/GIIR/mra'
    #'/home/k/PLAYGROUND10GB/SKULSTRIPpaper__'
    ]
import sys
[sys.path.insert(1,_) for _ in mylibpath]
del mylibpath

from tf_select_a_gpu import select_a_gpu
# select_gpu = gpus[gpu_id]
memory_limit = 32 #GB
select_a_gpu(gpus, gpu_id = 2, memory_limit=memory_limit)
# del gpu_id, select_a_gpu, select_gpu

# from DWT1DFB import DWT1D, IDWT1D
# from DWT2DFB import DWT2D, IDWT2D
import matplotlib.pyplot as plt

2025-05-24 22:02:06.562632: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748104326.583081   36719 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748104326.589366   36719 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-24 22:02:06.610762: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version 2.18.0
Num GPUs Available:  3
3 Physical GPUs available 
Selected 1 Logical GPU with 32 GB memory limit


I0000 00:00:1748104328.659830   36719 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 32768 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


In [2]:
mylibpath = [
    '/data1/kishoretarafdar/src.port/IIRbyoptimization.v0/IIRTF.v0'
    ]
import sys
[sys.path.insert(1,_) for _ in mylibpath]
del mylibpath

from IIR1Dv1 import IIR1D
from IIR2Dv1 import IIR2D
from IIR3Dv1 import IIR3D

In [3]:
# Dummy data
batch_size, N, channels, filters = 10, 128, 3, 2
x_train = tf.random.normal((batch_size, N, channels))
y_dummy = tf.random.normal((batch_size, N, filters))
# y_dummy = tf.zeros_like(x_train)
x_train.shape, y_dummy.shape, x_train.dtype, y_dummy.dtype

2025-06-17 21:25:41.939717: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-06-17 21:25:41.939772: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:137] retrieving CUDA diagnostic information for host: meherangarh
2025-06-17 21:25:41.939779: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:144] hostname: meherangarh
2025-06-17 21:25:41.939956: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:168] libcuda reported version is: 570.148.8
2025-06-17 21:25:41.939984: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:172] kernel reported version is: 570.148.8
2025-06-17 21:25:41.939989: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:259] kernel version seems to match DSO: 570.148.8


(TensorShape([10, 128, 3]), TensorShape([10, 128, 2]), tf.float32, tf.float32)

In [4]:
def model(input_shape=(256, 1)):
    inputs = tf.keras.layers.Input(input_shape)
    q = IIR1D(Delays=40, filters=2, tolerance=1e-8, max_steps=5000, local_lr=0.001,)(inputs)
    # print('q', q.shape)
    q = IIR1D(Delays=40, filters=2, tolerance=1e-8, max_steps=5000, local_lr=0.001,)(q)
    outputs = q  
    model = tf.keras.Model(inputs=[inputs], outputs=[outputs])
    return model

# N, channels = 64, 2
m = model(input_shape=(N, channels))#, Delays=Delays, filters=filters)  # (N=20, channels=16)
# m.compile(optimizer='adam', loss='mse')

# model = LayerModel(layer)
m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse', jit_compile=False)  # Disable XLA for debugging)
m.summary()

Tensor("while/Placeholder:0", shape=(), dtype=int32), Tensor("while/Placeholder:0", shape=(), dtype=int32), 

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 3)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ iir1d (IIR1D)                   │ (None, 128, 2)         │           486 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ iir1d_1 (IIR1D)                 │ (None, 128, 2)         │           324 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 810 (3.16 KB)

 Trainable params: 810 (3.16 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Run for 2 epochs
m.fit(x=x_train, y=y_dummy, epochs=10, batch_size=batch_size)

Epoch 1/10


Tensor("functional_1/iir1d_1/while/Placeholder:0", shape=(), dtype=int32), Tensor("functional_1/iir1d_1_2/while/Placeholder:0", shape=(), dtype=int32), 

/data1/kishoretarafdar/miniforge3/envs/tf218/lib/python3.12/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor']. Received: the structure of inputs=*
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 101s 101s/step - loss: 1.1385ional_1/iir1d_1_2/while/Placeholder:0", shape=(), dtype=int32)
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 93s 93s/step - loss: 1.0239
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 91s 91s/step - loss: 0.9854
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 89s 89s/step - loss: 0.9705
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 89s 89s/step - loss: 0.9615
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 88s 88s/step - loss: 0.9557
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 87s 87s/step - loss: 0.9544
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 87s 87s/step - loss: 0.9442
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 89s 89s/step - loss: 0.9449
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 87s 87s/step - loss: 0.9352
